# Psychological-Bias-Transfer Colab PilotRun these cells top-to-bottom. Do not skip the validator gate.

## Step 0: Clone / pull repoThis cell is safe to rerun.

In [ ]:
import os
from google.colab import userdata
REPO = userdata.get("REPO_URL", "https://github.com/coenhewes/Psychological-Bias-Transfer.git")
!if [ -d "Psychological-Bias-Transfer/.git" ]; then cd Psychological-Bias-Transfer && git pull "$REPO"; else git clone "$REPO"; fi
%cd Psychological-Bias-Transfer

## Step 1: Install dependenciesUse the repo `requirements.txt`. This cell is idempotent.

In [ ]:
!pip install -r requirements.txt

## Step 2: SecretsStore these in Colab: **Settings → Secrets**.

In [ ]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
JUDGE_BACKEND = userdata.get("JUDGE_BACKEND", "minimax")
JUDGE_MODEL = userdata.get("JUDGE_MODEL", "minimax-m3")
MINIMAX_API_KEY = userdata.get("MINIMAX_API_KEY", "")
ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY", "")

# Expose as plain env vars for libraries that expect them
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["MINIMAX_API_KEY"] = MINIMAX_API_KEY
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

## Step 3: Discover Dreaddit subredditsShow what is actually in `andreagasparini/dreaddit` so the next step uses real names.

In [ ]:
from datasets import load_dataset
from collections import Counter

print('Loading HF dataset...')
ds = load_dataset('andreagasparini/dreaddit', split='train')
print('columns=', ds.column_names)
print('num_rows=', len(ds))
print('subreddits top=')
c = Counter(str(x).lower() for x in ds['subreddit'])
for k, v in c.most_common(20):
    print(f"{k}\t{v}")

## Step 4: Build corporaPaste real subreddit names from the cell above. This uses MiniMax-backed matching internally until you customize further.

In [ ]:
%%bash
python3 data/corpus_builder.py \
  --source hf_dataset \
  --hf-dataset-id andreagasparini/dreaddit \
  --treatment-subreddits anxiety ptsd \
  --control-candidates assistance stress food_pantry \
  --out-dir data/processed \
  --target-tokens 1000000

## Step 5: Validate corpora (ABSOLUTE GATE)
If `gate_status` is not `PASS`, fix the corpora. Do not train until this passes.

In [ ]:
%%bash
python3 data/corpus_validator.py \
  --treatment data/processed/treatment_corpus.jsonl \
  --control data/processed/control_corpus.jsonl \
  --out-dir data/validation

## Step 6: Pilot finetuneTuneable values are `--model`, `--corpus`, `--seed`, and the optional `--config` path.

In [ ]:
%%bash
python3 training/finetune_qlora.py --model meta-llama/Meta-Llama-3.1-8B-Instruct --corpus treatment --seed 42 --config config/training_config.yaml

## Step 7: Generate eval outputs

In [ ]:
%%bash
python3 evaluation/generate_outputs.py \
  --base-model meta-llama/Meta-Llama-3.1-8B-Instruct \
  --adapter checkpoints/treatment_seed42/final_adapter \
  --condition-name treatment_seed42 \
  --out data/generations/treatment_seed42.jsonl

## Step 8: Judge outputs with MiniMax-M3

In [ ]:
%%bash
python3 evaluation/judge.py \
  --generations data/generations/treatment_seed42.jsonl \
  --judge minimax --model minimax-m3 \
  --out data/judged/treatment_seed42.jsonl

## Step 9: Analysis

In [ ]:
%%bash
python3 analysis/statistical_analysis.py \
  --judged-dir data/judged \
  --out-dir results \
  --corpus-hit-rate-report data/validation/validation_report.json

## Step 10: Release artifacts

In [ ]:
%%bash
python3 scripts/build_release_artifacts.py \
  --processed-dir data/processed \
  --validation-dir data/validation \
  --results-dir results \
  --out-dir data/release